# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jahnzaibakhtar/Flyrank-ml-internship-starter/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

In [ ]:
I'm using Logistic Regression, then comparing against a Random Forest, for a ranking question
("which items first?") evaluated at precision@K rather than plain accuracy. Logistic Regression
fits first because it's readable — I can name which features push the score up or down, which
matters for a queue an editor has to trust. Random Forest is the second model to check whether
added complexity actually earns its place; I don't add it just because it's stronger by default.

Target: is_declining_label — used here for the first time as a training label (not a feature).
It's a rule-derived label, not a fully independent observed outcome, so results are decision-
support signal, not proof of true future decline.

Features: the same trailing, same-day-knowable signals from earlier weeks — ctr, avg_position
(with a missingness flag, not a blind fillna), scroll_rate, ai_traffic_pct, days_since_update,
content_type — excluding trend_direction and trend_pct, which the label is computed from.

In [ ]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

df["avg_position_missing"] = (df["avg_position"] == 0).astype(int)  # has_-flag, not fillna(0)
df["avg_position_clean"] = df["avg_position"].replace(0, df["avg_position"].median())

df = pd.get_dummies(df, columns=["content_type"], prefix="ctype")
ctype_cols = [c for c in df.columns if c.startswith("ctype_")]

X_cols = ["ctr", "avg_position_clean", "avg_position_missing", "scroll_rate",
          "ai_traffic_pct", "days_since_update"] + ctype_cols
X = df[X_cols].fillna(0)
y = df["is_declining_label"]
groups = df["client_id"]

print(f"Feature matrix: {X.shape}")
print(f"Label base rate: {y.mean():.1%}")

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [ ]:
I use a GROUPED split by client_id, not a random row split or a time split. Content items from
the same client share context (industry, baseline traffic level), so a random row split would
let the model see near-duplicate client context in both train and test, inflating the score.
client_id is meant for grouping/splitting only, per the data's own rules — this is that use.

I'm not doing a time-aware split here because the starter CSV is a single trailing-90-day
snapshot per item, not a multi-period panel — there's no future window to hold out within this
dataset. That's a limitation I name in Section 4, not something this split can fix.

In [ ]:
from sklearn.model_selection import GroupShuffleSplit

splitter = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(splitter.split(X, y, groups=groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

train_clients = set(groups.iloc[train_idx])
test_clients = set(groups.iloc[test_idx])
print(f"Train: {len(X_train)} rows, {len(train_clients)} clients")
print(f"Test: {len(X_test)} rows, {len(test_clients)} clients")
print(f"Client overlap between train/test (should be 0): {len(train_clients & test_clients)}")
print(f"Test set label base rate: {y_test.mean():.1%}")

## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [ ]:
Same test split, same precision@K metric, same base rate as my Week-4 baseline — the baseline is
recomputed here on this exact split so the comparison is fair, not pulled from a different run.

In [ ]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

test_df = df.iloc[test_idx]
stale = (test_df["days_since_update"] >= 180).astype(int)
underperforming = (test_df["ctr"] < test_df["ctr"].median()).astype(int)
baseline_score = (stale * underperforming).values

logreg = LogisticRegression(max_iter=1000, random_state=42)
logreg.fit(X_train, y_train)
logreg_score = logreg.predict_proba(X_test)[:, 1]

rf = RandomForestClassifier(n_estimators=200, max_depth=6, random_state=42)
rf.fit(X_train, y_train)
rf_score = rf.predict_proba(X_test)[:, 1]

base_rate = y_test.mean()
rows = []
for k in [10, 20, 50]:
    rows.append({
        "k": k,
        "base_rate": round(base_rate, 3),
        "baseline_precision": round(precision_at_k(baseline_score, y_test.values, k), 3),
        "logreg_precision": round(precision_at_k(logreg_score, y_test.values, k), 3),
        "rf_precision": round(precision_at_k(rf_score, y_test.values, k), 3),
    })

comparison_table = pd.DataFrame(rows)
print(comparison_table)

import os
os.makedirs("work/outputs", exist_ok=True)
comparison_table.to_json("work/outputs/w05_comparison_table.json", orient="records", indent=2)

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [ ]:
[State plainly which model wins and at which K — e.g. "Random Forest beats the baseline and
Logistic Regression at precision@10 and @20, but the three are roughly tied at @50 — added
complexity earns its place at the top of the queue but not further down."]

Top features: [name the top 2-3 from the printed importances and say whether each makes sense —
e.g. "ctr is the top feature, plausible since low relative CTR is a direct engagement signal. If
any feature looked suspiciously dominant, that's the leakage smell to check before trusting it —
none did here."]

3 concrete wrong cases: [after inspecting false negatives, name what they have in common — e.g.
"the model's misses are mostly top-5-position items with strong position but quietly falling CTR
— position alone reads as healthy even as engagement erodes underneath it."]

In [ ]:
importances = pd.Series(rf.feature_importances_, index=X_cols).sort_values(ascending=False)
print("Top features (fit-based importance):")
print(importances.head(5))

from sklearn.inspection import permutation_importance
perm = permutation_importance(rf, X_test, y_test, n_repeats=10, random_state=42)
perm_importances = pd.Series(perm.importances_mean, index=X_cols).sort_values(ascending=False)
print("\nTop features (permutation importance — more trustworthy):")
print(perm_importances.head(5))

test_results = test_df.copy()
test_results["rf_score"] = rf_score
test_results["true_label"] = y_test.values
false_negatives = test_results[
    (test_results["true_label"] == 1) & (test_results["rf_score"] < 0.3)
].sort_values("rf_score")

print(f"\nFalse negatives (real decline, low score): {len(false_negatives)}")
print(false_negatives[["content_id", "avg_position", "ctr", "days_since_update", "rf_score"]].head(3))

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.